# CardioIA — IA em Séries Temporais: Regressão Logística vs. LIF

**Objetivo:** Comparar o desempenho e explicabilidade de dois classificadores para detectar taquicardia em séries temporais de BPM (Batimentos por Minuto).
1. **Regressão Logística:** Modelo estatístico tradicional, altamente explicável.
2. **Modelo Neuromórfico LIF (Leaky Integrate-and-Fire):** Abordagem bio-inspirada que processa informações de forma análoga a redes neurais biológicas.

In [ ]:
# Instalação e imports
!pip install -q numpy pandas scikit-learn matplotlib seaborn
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import time

# Configuração de estilo e paleta médica
sns.set_theme(style="whitegrid")
paleta_medica = {0: '#38bdf8', 1: '#ef4444'} # Azul para Normal, Vermelho para Taquicardia
os.makedirs('docs/images', exist_ok=True)

In [ ]:
# Geração dos dados simulados
np.random.seed(42)
N_SAMPLES = 500

# Gerar labels: ~80% normal (0), ~20% taquicardia (1)
y = np.random.choice([0, 1], size=N_SAMPLES, p=[0.8, 0.2])

# Gerar features com base nas labels
bpm_atual = np.where(y == 0, np.random.normal(80, 15, N_SAMPLES), np.random.normal(135, 10, N_SAMPLES))
bpm_media_5s = bpm_atual + np.random.normal(0, 5, N_SAMPLES)
bpm_std_5s = np.where(y == 0, np.random.normal(3, 1, N_SAMPLES), np.random.normal(8, 2, N_SAMPLES))
temperatura = np.where(y == 0, np.random.normal(36.5, 0.5, N_SAMPLES), np.random.normal(37.8, 0.8, N_SAMPLES))
umidade = np.random.normal(50, 10, N_SAMPLES) # Umidade afeta menos diretamente

# Adicionar ruído gaussiano para simular sensor real
noise = np.random.normal(0, 2, N_SAMPLES)
bpm_atual += noise

# Montar DataFrame
data = pd.DataFrame({
    'bpm_atual': bpm_atual,
    'bpm_media_5s': bpm_media_5s,
    'bpm_std_5s': bpm_std_5s,
    'temperatura': temperatura,
    'umidade': umidade,
    'label': y
})

print("Formato dos dados:", data.shape)
print("\nDistribuição das classes:")
print(data['label'].value_counts(normalize=True))

In [ ]:
# Análise Exploratória de Dados (EDA)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribuição das classes
sns.countplot(x='label', data=data, palette=paleta_medica, ax=axes[0])
axes[0].set_title("Distribuição das Classes (0=Normal, 1=Taquicardia)")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Contagem")

# Histograma de BPM Atual
sns.histplot(data=data, x='bpm_atual', hue='label', palette=paleta_medica, kde=True, ax=axes[1])
axes[1].set_title("Distribuição de BPM Atual por Classe")
axes[1].set_xlabel("BPM")
axes[1].set_ylabel("Frequência")

# Matriz de Correlação
corr = data.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', ax=axes[2], fmt=".2f")
axes[2].set_title("Matriz de Correlação das Features")

plt.tight_layout()
plt.show()

In [ ]:
# Pré-processamento: Separação Treino/Teste e Padronização
X = data.drop('label', axis=1)
y = data['label']

# Split 80/20 estratificado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Padronização (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Tamanho do conjunto de treino:", X_train_scaled.shape)
print("Tamanho do conjunto de teste:", X_test_scaled.shape)

In [ ]:
# MODELO 1: Regressão Logística
inicio_lr = time.time()
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predições
lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
tempo_lr = (time.time() - inicio_lr) * 1000 # ms

# Métricas
lr_acc = accuracy_score(y_test, lr_preds)
lr_prec = precision_score(y_test, lr_preds)
lr_rec = recall_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds)
lr_auc = roc_auc_score(y_test, lr_probs)

print("--- Regressão Logística ---")
print(f"Accuracy:  {lr_acc:.4f}")
print(f"Precision: {lr_prec:.4f}")
print(f"Recall:    {lr_rec:.4f}")
print(f"F1-Score:  {lr_f1:.4f}")
print(f"ROC AUC:   {lr_auc:.4f}")

In [ ]:
# MODELO 2: Rede Neuromórfica LIF (Leaky Integrate-and-Fire)
# Implementação puramente com NumPy
inicio_lif = time.time()

class LIFNeuron:
    def __init__(self, tau=20.0, V_rest=-70.0, V_thresh=-55.0, V_reset=-75.0, R=1.0, dt=1.0):
        self.tau = tau
        self.V_rest = V_rest
        self.V_thresh = V_thresh
        self.V_reset = V_reset
        self.R = R
        self.dt = dt
        self.V = V_rest
        
    def step(self, current):
        # Equação LIF: tau * dV/dt = -(V - V_rest) + R * I(t)
        dV = (-(self.V - self.V_rest) + self.R * current) * (self.dt / self.tau)
        self.V += dV
        
        if self.V >= self.V_thresh:
            self.V = self.V_reset
            return 1 # Disparo (Spike)
        return 0

# Para a classificação LIF, converteremos as features numa "corrente de entrada" I(t)
# Usaremos os pesos da Regressão Logística como conexão sináptica representativa
pesos = lr_model.coef_[0]
bias = lr_model.intercept_[0]

lif_probs = []
lif_preds = []

# Simulação da rede neuromórfica no tempo discreto
T_SIMULACAO = 100 # ms
for i in range(len(X_test_scaled)):
    corrente_base = np.dot(X_test_scaled[i], pesos) + bias
    
    # Mapear corrente base para o neurônio: valores negativos cortados (ReLU-like), escalonados
    I_inj = np.maximum(0, corrente_base * 20.0) 
    
    neuronio = LIFNeuron()
    spikes = 0
    for t in range(T_SIMULACAO):
        spikes += neuronio.step(I_inj)
    
    # Taxa de disparo (firing rate) como proxy para probabilidade
    firing_rate = spikes / T_SIMULACAO
    lif_probs.append(firing_rate)
    
    # Classificação limiar: se houver taxa relevante de disparos, é taquicardia
    lif_preds.append(1 if firing_rate > 0.05 else 0)

tempo_lif = (time.time() - inicio_lif) * 1000 # ms

# Normalizar pseudo-probabilidades para o ROC
lif_probs = np.array(lif_probs)
lif_probs = lif_probs / max(lif_probs) if max(lif_probs) > 0 else lif_probs

# Métricas LIF
lif_acc = accuracy_score(y_test, lif_preds)
lif_prec = precision_score(y_test, lif_preds, zero_division=0)
lif_rec = recall_score(y_test, lif_preds, zero_division=0)
lif_f1 = f1_score(y_test, lif_preds, zero_division=0)
lif_auc = roc_auc_score(y_test, lif_probs)

print("--- Modelo LIF (Neuromórfico) ---")
print(f"Accuracy:  {lif_acc:.4f}")
print(f"Precision: {lif_prec:.4f}")
print(f"Recall:    {lif_rec:.4f}")
print(f"F1-Score:  {lif_f1:.4f}")
print(f"ROC AUC:   {lif_auc:.4f}")

In [ ]:
# Comparação Lado a Lado dos Modelos

# 1. Tabela Comparativa
resumo = pd.DataFrame({
    'Modelo': ['Regressão Logística', 'LIF Neuromórfico'],
    'Accuracy': [lr_acc, lif_acc],
    'Precision': [lr_prec, lif_prec],
    'Recall': [lr_rec, lif_rec],
    'F1': [lr_f1, lif_f1],
    'AUC': [lr_auc, lif_auc],
    'Tempo(ms)': [tempo_lr, tempo_lif]
})
display(resumo)

# 2. Gráfico de Barras Comparativo
resumo_melted = resumo.melt(id_vars='Modelo', value_vars=['Accuracy', 'Precision', 'Recall', 'F1', 'AUC'], var_name='Métrica', value_name='Valor')

plt.figure(figsize=(10, 6))
sns.barplot(x='Métrica', y='Valor', hue='Modelo', data=resumo_melted, palette=['#38bdf8', '#ef4444'])
plt.title("Comparação de Métricas de Desempenho")
plt.ylim(0, 1.1)
plt.legend(loc='lower right')
plt.savefig('docs/images/ia_comparacao_modelos.png')
plt.show()

# 3. Matrizes de Confusão
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_test, lr_preds), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title("Matriz de Confusão - Regressão Logística")
axes[0].set_xlabel("Predito")
axes[0].set_ylabel("Real")

sns.heatmap(confusion_matrix(y_test, lif_preds), annot=True, fmt='d', cmap='Reds', ax=axes[1])
axes[1].set_title("Matriz de Confusão - LIF Neuromórfico")
axes[1].set_xlabel("Predito")
axes[1].set_ylabel("Real")
plt.tight_layout()
plt.savefig('docs/images/ia_matriz_confusao.png')
plt.show()

# 4. Curvas ROC Sobrepostas
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_lif, tpr_lif, _ = roc_curve(y_test, lif_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, color='#38bdf8', lw=2, label=f'Reg. Logística (AUC = {lr_auc:.3f})')
plt.plot(fpr_lif, tpr_lif, color='#ef4444', lw=2, label=f'LIF Neuromórfico (AUC = {lif_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.title("Curvas ROC Comparativas")
plt.xlabel("Taxa de Falso Positivo")
plt.ylabel("Taxa de Verdadeiro Positivo")
plt.legend(loc='lower right')
plt.savefig('docs/images/ia_curvas_roc.png')
plt.show()

## Análise Crítica

1. **Desempenho (Qual modelo acerta mais?):**
   A **Regressão Logística** apresenta métricas de precisão e acurácia superiores (quase perfeitas) em relação ao LIF. Isso ocorre pois as *features* geradas possuem uma fronteira de decisão amplamente linear (taquicardia definida em > 120 BPM com ruído moderado). O **LIF**, por simular as disparadas de *spikes* temporalmente ($dV/dt$), embute perdas de resolução heurísticas que resultam em uma margem maior de falsos negativos em zonas limiares.

2. **Explicabilidade e Transparência Clínica:**
   A **Regressão Logística** é o padrão-ouro de transparência estatística na saúde: os pesos ($eta$) de cada atributo podem ser diretamente compreendidos por um médico. Em contraste, o modelo **LIF** age como uma *caixa bio-inspirada*; seu diagnóstico provém da agregação de voltagens de membrana em janelas de 100ms, o que impossibilita a explicação dedutiva sintética exigida em auditorias clínicas.

3. **Trade-offs: Precisão vs. Transparência vs. Custo Computacional:**
   A Regressão exige o simples cálculo de um produto escalar ($\sim O(N)$) em microssegundos. Já a simulação LIF precisa iterar as integrais diferenciais a cada instante `dt`, disparando radicalmente o tempo computacional em CPUs tradicionais.

4. **Aplicabilidade em Dispositivos Embarcados (ESP32):**
   Exportar os coeficientes de uma **Regressão Logística** para inferência via código C/C++ no **ESP32** requer apenas multiplicações contínuas, sendo excelente para abordagens *Edge AI*. A modelagem **LIF** clássica é inexequível em MCUs comuns devido ao peso dos laços temporais, ganhando sentido exclusivamente caso a arquitetura de borda incorpore chips hardware dedicados (ex: processadores neuromórficos Spiking Neural Networks - SNNs).

## Conclusão
No escopo do projeto **CardioIA — FIAP Fase 3**, evidenciamos que nem sempre a modelagem mais avançada e bio-inspirada é a mais adequada para aplicações de Telemedicina via IoT.
O comparativo provou que a **Regressão Logística** atende de modo superior aos pilares de confiabilidade médica: alta acurácia, baixo viés computacional para execução na *borda* (Edge Computing) e clareza explicativa absoluta. 
A técnica neuromórfica (LIF), embora revolucionária conceitualmente para processamento analógico temporal de sensores contínuos, apresenta um *trade-off* desfavorável em microcontroladores puramente lógicos como o ESP32 sem hardware acelerador dedicado.